# 11 - Offline joint-training reference

Train one fresh model on the pooled training data from every task, then evaluate it on the same task groups as the sequential experiments. This measures an offline reference for the cost of sequential data access.

Select **TensorFlow 2.20 (Docker GPU)**, restart the kernel, then run top to bottom. These notebooks create supplemental reference results; no measured results are included yet.

## 1. Check the runtime

The maintained environment is TensorFlow 2.20 with Keras 3. Run this notebook after other GPU training has finished.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "semantic_consolidation/config.py").is_file())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("TF_FORCE_GPU_ALLOW_GROWTH", "true")

from IPython.display import display
from notebooks.thesis.workflow import check_runtime
print(check_runtime())
from notebooks.thesis.reference_benchmarks import (
    configure_reference, prepare_reference, train_reference, finish_reference,
)

## 2. Select a paired experiment

The default is CIFAR-10, seed 17, with the current development validation split. Choose CIFAR-100 in the same cell to run its reference. Use the same fixed seed and recipe as the run being compared.

Use `test` only after the recipe is fixed. Such a test run remains a supplemental, unregistered reference; it is not part of the frozen 24-run campaign.

In [ ]:
DATASET = "cifar10"  # "cifar10" or "cifar100"
SEED = 17  # Match the comparison run's seed and class order.
EVALUATION_SPLIT = "validation"  # Set "test" only after fixing the recipe.

In [ ]:
BENCHMARK = 'offline_joint'
config = configure_reference(
    DATASET, BENCHMARK, seed=SEED, evaluation_split=EVALUATION_SPLIT,
)

## 3. Review the training plan

All permitted training rows are available from the start and are mixed across tasks in one fit. The task groups are retained for evaluation; they are not training stages. No replay, stored-example buffer, knowledge distillation, or semantic route is used.

Both references retain the thesis DiT architecture and combined diffusion/classification objective. Current recipes use 40 epochs for CIFAR-10 and 60 for CIFAR-100, at batch size 64. These are shared nominal passes per permitted training example; they do not establish equal compute with the replay platform. The displayed configuration is authoritative.

In [ ]:
schedule = config.continually_learn
display({
    "dataset": config.dataset.name,
    "reference": BENCHMARK,
    "seed": schedule.seed,
    "evaluation_split": EVALUATION_SPLIT,
    "epochs_per_training_example": config.training.epochs,
    "batch_size": config.dataset.batch_size,
    "class_order": schedule.class_order,
    "task_groups": schedule.task_groups,
})

## 4. Prepare data, model, and result folder

The helper applies the paired seed, fixed schedule, and held-out split. Do not change the configuration after preparation.

In [ ]:
context = prepare_reference(config)
print("Results folder:", context["run_dir"])

## 5. Train once

This is the full reference run and may take substantial time. A repeated training-cell execution is rejected for the same prepared context. For a fresh experiment, restart the kernel and run from the beginning.

In [ ]:
# The helper refuses a second training call for the same prepared context.
history = train_reference(config, context)

## 6. Save and inspect results

Inspect final overall accuracy and the per-task evaluation table. **Forgetting and backward transfer are not applicable**: this model has no sequence of task-end states. They must remain unavailable, not be reported as zero.

In [ ]:
summary, per_task = finish_reference(config, context, history)
display(summary)
display(per_task)
print("Saved results:", context["run_dir"])

## Interpretation and cleanup

Treat this result as an empirical offline upper reference for the selected model and recipe. It is not the maximum possible accuracy, and another method can exceed it. The pooled model sees future tasks immediately, so it cannot establish the performance of an online learner.

After the final cell succeeds, **save this notebook with its outputs, then restart its kernel** to release its training state before running another notebook. Result artifacts are saved separately in the printed folder.

See [benchmark definitions, comparison limits, and sources](BENCHMARK_REFERENCES.md).